# Goal

Первое прогон модели против 8 обзов из 33. Выбираем последние 4 обза (высокочастостники) и ещё добавляем 4 обза с интервалом по 1 секунде. Таким образом имеются как текущие наблюдения, так и удалённые. При этом 8 обзов позволяют иметь в параллель 6 запусков.

В отличие от `18n_study_37.1` тут используется ванильный, переплетенный (interleaved) вариант.

# TARGET_NOTEBOOK_FNAME

In [1]:
TARGET_NOTEBOOK_FNAME = '18n_ppo_tr_frostbite_08.ipynb'

# GRID_SEARCH_SPACE

In [2]:
# @launchit.collect
# GRID_SEARCH_SPACE = dict(
# )

# set_hyperparameters

In [3]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.general.random_seed = random.randint(1, 100)
    HP.general.is_torch_deterministic = True
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True
    HP.general.is_torch_amp_on_rollout = True
    
    HP.env.count = 32 
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6

    parent = '18d_world_model_09:183'
    HP.vision_head.parent = dict(model=parent, weights=parent)
    HP.vision_head.is_trainable = False
    
    HP.encoder.parent = dict(model=parent, weights=parent)
    HP.encoder.is_trainable = True

    HP.agent.parent = None 
    HP.agent.sequence_length = 33
    HP.agent.sequence_ages = [0, 1, 2, 3, 8, 16, 24, 32]
    HP.agent.action_plan_length = 10
    HP.agent.d_model = 256 
    HP.agent.transformer = dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    HP.agent.is_trainable = True
    
    # Test params
    HP.test.env_rams = [
        'com.develorium.neurolab.frostbite_ram:level1:1:cls=none',
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none',
    ]
    HP.test.env_ram_patches = [
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_right_of_igloo', 'bear_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_left_of_igloo', 'bear_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_right_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_20', 'bailey_near_center'], # *
    ]
    HP.test.break_on_level_passed = True
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = 3_000_000 # total number of steps 
    HP.ppo.rollout_steps_count = 512 # how many steps to run in a single policy rolllout
    HP.ppo.rollout_env_rams = [
        'com.develorium.neurolab.frostbite_ram:level1:1:cls=none',
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none',
    ]
    HP.ppo.rollout_env_ram_patches = [
        ['no_score', 'last_life', 'full_igloo',            'temperature_10', 'bailey_right_at_the_igloo_door'], # 0 
        ['no_score', 'last_life', 'full_igloo',            'temperature_10', 'bailey_very_near_igloo_door'], # 1
        ['no_score', 'last_life', 'full_igloo',            'temperature_10', 'bailey_near_center'], # 2
        ['no_score', 'last_life', 'one_remaining_igloo',   'temperature_10', 'bailey_near_center'], # 3
        ['no_score', 'last_life', 'three_remaining_igloo', 'temperature_20', 'bailey_near_center', ], # 4
        ['no_score', 'last_life', 'half_igloo', 'bailey_near_center'], # 5
        ['no_score', 'last_life', 'half_igloo'], # 6

        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_right_of_igloo', 'bear_to_the_left_of_igloo'], # 7
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_left_of_igloo', 'bear_to_the_left_of_igloo'], # 8
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_right_of_igloo'], # 9
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_left_of_igloo'], # 10
        ['no_score', 'last_life', 'full_igloo', 'temperature_20', 'bailey_near_center'], # 11
    ]
    HP.ppo.rollout_env_stories = [
        '0;0:7',  # level 1 - teach agent to enter complete igloo
        '1;7:12', # level 5 - teach agent to enter complete blinking igloo
    ]
    
    HP.ppo.epochs_count = 2 
    HP.ppo.batch_size = 256 
    HP.ppo.learn_rate = 'const(0.00025)'
    HP.ppo.optimizer = 'AdamW'

    HP.ppo.value_loss = dict(coef=0.2)
    HP.ppo.entropy_loss = dict(coef='const(0.05)')
    HP.ppo.consistency_loss = dict(type='MSE', coef=0.1)
    HP.ppo.prediction_loss = dict(coef=0.1)
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # the target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization
    return HP

# Results

`38.1` — первый прогон новой модели `18n_ppo_tr_frostbite_08` (ванильный, переплетенный/interleaved вариант) на окне из 8 обзов: `sequence_ages=[0, 1, 2, 3, 8, 16, 24, 32]` (последние 4 + 4 с интервалом 1 секунда). Родитель фиксирован — `18d_world_model_09:183` (frozen `vision_head` + trainable `encoder`), агент с нуля. Базовое сравнение — с `35.1` (шаг 1 суперисследования 35, `frostbite_07`, окно из 4 обзов, `is_obs_usage_enforced=True`) и `31.1` (`frostbite_07`, окно из 4 обзов, родители-руки `:183`/`:185`). Идентично: родитель `:183`, `action_plan_length=10`, 3M шагов, 32 env, `learn_rate=const(0.00025)`, `ent_coef=const(0.05)`, `vf_coef=0.2`, `consistency=MSE 0.1`, `prediction=0.1`, `gamma=0.997`, тест (уровни 1+5, `break_on_level_passed=True`), истории `0;0:7` и `1;7:12`. Отличается: целевая модель (08 против 07), окно (8 против 4 обзов), `batch_size=256` (против 512), `is_torch_amp_on_rollout=True` (в `35.1`/`31.1` False), у `35.1` дополнительно `is_obs_usage_enforced=True` — смена мультипараметрическая, атрибуция дельты на один фактор невозможна.

**Серии: `38.1` vs `35.1` vs `31.1`** (метрика — `test/levels_passed_mean`)

| Metric | Experiment `18n_study_38.1` (08, 8 обзов, interleaved) | Experiment `18n_study_35.1` (07, 4 обза, obs-enforced) | Experiment `18n_study_31.1` (07, 4 обза, руки :183/:185) | Statistical Result / Verdict |
| :--- | :--- | :--- | :--- | :--- |
| **Sample Size (n)** | 18 | 18 | 24 | — |
| **Mean Score** | 0.6257 | 0.6583 | 0.6272 | -0.0326 vs `35.1`, -0.0015 vs `31.1` (н/з) |
| **Best Value** | 0.7875 (`57`) | 0.896875 (`956`) | 0.796875 (`521`, `528`) | — |
| **Standard Deviation** | 0.1001 | 0.1446 | 0.0881 | шум у `38.1` меньше, чем у `35.1` (разброс `35.1` раздут аутлиером `945` — 0.34375) |
| **Shapiro-Wilk (p-value)** | 0.7262 (нормальное) | 0.8389 (нормальное) | 0.1066 (нормальное) | все три распределения близки к нормальному |
| **Welch's t-test (p-value)** | — | — | — | **0.4372** (не значимо, `38.1` vs `35.1`); **0.9595** (не значимо, `38.1` vs `31.1`) |
| **Mann-Whitney U (p-value)** | — | — | — | **0.3501** (не значимо); **0.9696** (не значимо) |
| **Cohen's d (Effect Size)** | — | — | — | **-0.2625** (малый, в пользу `35.1`); **-0.0163** (пренебрежимо малый, vs `31.1`) |

Для контекста — `37.1` (то же окно из 8 обзов, вариант, отличные от ванильного interleaved; 9 из 13 триалов завершено): лучший 0.790625 (`52`), mean 0.5944 — тот же уровень, что `38.1`: interleaved и прочий вариант по решающему метрику неразличимы.

Системная проблема `recon/0.1+recon/3.0`: в журнале только решающий метрик — **не проверено по журналу**.

**Выводы**
1) Новая 8-обзовая window `frostbite_08` на шаге 1 по решающему метрику не теряет: `38.1` (mean 0.6257) статистически неотличима и от `35.1` (Welch p=0.4372, MWU p=0.3501, d≈-0.26), и от `31.1` (Welch p=0.9595, MWU p=0.9696, d≈-0.02) — но и прироста нет.
2) Лучший триал `38.1` — `57` (0.7875): ниже лучшего `35.1` (`956`, 0.896875) и на волосок ниже лучших `31.1` (`521`/`528`, 0.796875); при этом шум у `38.1` ниже, чем у `35.1` (std 0.1001 против 0.1446).
3) `37.1` (прочий вариант того же окна) на том же уровне (`52` — 0.790625): выбор между interleaved и прочим вариантом по метрике не аргументируется — это вопрос дизайна.
4) Статус `recon/0.1+recon/3.0` — «не проверено по журналу» (в журнале только решающий метрик); вердикт по проблеме за Боссом.
5) Следующий шаг: чемпион `57` (0.7875), при необходимости с `63` (0.759375) — PARENTS для шага 2 новой цепи.

# System

In [4]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [5]:
CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = json.load(connection_file).get('jupyter_session')
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_08.ipynb',
 'target_notebook_name': '18n_ppo_tr_frostbite_08',
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_38.1/18n_study_38.1.ipynb',
 'optuna_study_name': '18n_study_38.1',
 'optuna_study_serial': '38.1',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_38.1/18n_study_38.1.optuna'}

In [6]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)

In [7]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

In [8]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
        OPTUNA_DECISIVE_METRIC='test/levels_passed_mean',
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

In [9]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [10]:
optuna_study = optuna.create_study(
    study_name=CONFIG.optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
    load_if_exists=True,
)
optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
launches_count = 18
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while launches_count is None or completed_launches_count < launches_count:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if launches_count is None or (completed_launches_count + len(futures) < launches_count):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

[I 2026-10-11 18:31:36,791] A new study created in Journal with name: 18n_study_38.1


2026.10.11-18:31:37.093588     0.270 >> Model instance registered, version=56
2026.10.11-18:31:37.107187     0.004 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_08-launch56.ipynb"
2026.10.11-18:31:37.107396     0.004 >> 6.0 idle runners exist, submitted launch "18n_ppo_tr_frostbite_08:56"; running launches=1
2026.10.11-18:32:08.113491     0.283 >> Model instance registered, version=57
2026.10.11-18:32:08.122110     0.004 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_08-launch57.ipynb"
2026.10.11-18:32:08.122308     0.004 >> 5.3 idle runners exist, submitted launch "18n_ppo_tr_frostbite_08:57"; running launches=2
2026.10.11-18:32:39.116350     0.270 >> Model instance registered, version=58
2026.10.11-18:32:39.129558     0.004 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_08-launch58.ipynb"
2026.10.11-18:32:39.129760     0.004 >> 4.3 idle runners exist, submitted launch "18n_ppo_tr_frostbite_08:58"; ru

In [11]:
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs.get('MODEL_VERSION', 'n/a')}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

[I 2026-10-11 22:42:06,678] Using an existing study with name '18n_study_38.1' instead of creating a new one.


2026.10.11-22:42:06.680303     5.032 >> Study statistics: 
2026.10.11-22:42:06.683022     0.003 >> 	Number of finished trials: 18
2026.10.11-22:42:06.683839     0.001 >> 	Number of pruned trials: 0
2026.10.11-22:42:06.684701     0.001 >> 	Number of complete trials: 18
2026.10.11-22:42:06.686013     0.001 >> Best trial:
2026.10.11-22:42:06.686884     0.001 >> 	Value: 0.7875
2026.10.11-22:42:06.687339     0.000 >> 	Model version: 57
2026.10.11-22:42:06.687715     0.000 >> 	Params: 


In [12]:
top_trials = sorted(filter(lambda t: t.values, study.trials), key=lambda t: -t.values[0])[:3]
LOG(f'Top trials:')

for i, trial in enumerate(top_trials):
    LOG(f'{i+1}) {CONFIG.target_notebook_name + ':' + trial.user_attrs['MODEL_VERSION']}, value={trial.values[0]}')

2026.10.11-22:42:06.697102     0.009 >> Top trials:
2026.10.11-22:42:06.697818     0.001 >> 1) 18n_ppo_tr_frostbite_08:57, value=0.7875
2026.10.11-22:42:06.698265     0.000 >> 2) 18n_ppo_tr_frostbite_08:63, value=0.759375
2026.10.11-22:42:06.698561     0.000 >> 3) 18n_ppo_tr_frostbite_08:70, value=0.73125


In [13]:
values = []

for trial in study.trials:
    if trial.values is not None:
        values.append(trial.values[0])

values = np.array(values)
values.mean(), values.std()

(np.float64(0.6256944444444446), np.float64(0.0972693338235085))

In [14]:
','.join(map(str, values.tolist()))

'0.51875,0.7875,0.421875,0.671875,0.625,0.59375,0.640625,0.759375,0.684375,0.696875,0.553125,0.725,0.684375,0.5375,0.73125,0.51875,0.59375,0.51875'